# Добавляем рекламы

Объединяем метрики API из `conversion.json` с [рекламной выгрузкой](https://drive.google.com/file/d/12vCtGhJlcK_CBcs8ES3BfEPbk6OJ45Qj/view).
Сначала выполните этап расчёта метрик в `data_preparation.ipynb`, если нужно обновить конверсию.
Этот notebook запускается из корня репозитория. Рекламный CSV скачивается при каждом запуске.

Одна строка результата соответствует дню. Складываем визиты и регистрации по всем платформам,
рекламные затраты — по всем кампаниям дня. Если в день несколько кампаний, их уникальные
названия перечисляем через запятую: дневные метрики между кампаниями не распределяем.
Для дней без рекламы: `cost = 0`, `utm_campaign = "none"`.
Период результата определяется таблицей конверсий. Реклама за его пределами в итог не входит.

In [1]:
import pandas as pd
import requests

conversion = pd.read_json("./conversion.json", convert_dates=["date_group"])
response = requests.get(
    "https://drive.google.com/uc",
    params={"export": "download", "id": "12vCtGhJlcK_CBcs8ES3BfEPbk6OJ45Qj"},
    timeout=120,
)
response.raise_for_status()
expected_columns = ["date", "utm_source", "utm_medium", "utm_campaign", "cost"]
if response.content.decode("utf-8-sig").splitlines()[0].split(",") != expected_columns:
    raise ValueError("Ответ Google Drive не соответствует рекламному CSV")
with open("data/ads.csv", "wb") as target:
    target.write(response.content)
ads = pd.read_csv("data/ads.csv")
ads["date_group"] = pd.to_datetime(ads["date"], format="ISO8601", errors="raise").dt.normalize()
ads["cost"] = pd.to_numeric(ads["cost"], errors="raise")
if ads[["date_group", "cost", "utm_campaign"]].isna().any().any():
    raise ValueError("В рекламе отсутствуют даты, затраты или названия кампаний")
ads.head()

,date,utm_source,utm_medium,utm_campaign,cost,date_group
0,2023-03-01T10:54:41,google,cpc,advanced_algorithms_series,212,2023-03-01
1,2023-03-02T10:32:35,google,cpc,advanced_algorithms_series,252,2023-03-02
2,2023-03-03T19:21:40,google,cpc,advanced_algorithms_series,202,2023-03-03
3,2023-03-04T17:52:04,google,cpc,advanced_algorithms_series,223,2023-03-04
4,2023-03-05T05:35:13,google,cpc,advanced_algorithms_series,265,2023-03-05


In [2]:
def combine_advertising(conversion_data, advertising_data):
    daily_conversion = conversion_data.groupby("date_group", as_index=False).agg(
        visits=("visits", "sum"), registrations=("registrations", "sum"),
    )
    daily_ads = advertising_data.groupby("date_group", as_index=False).agg(
        cost=("cost", "sum"),
        utm_campaign=("utm_campaign", lambda names: ", ".join(sorted(names.unique()))),
    )
    result = daily_conversion.merge(
        daily_ads, on="date_group", how="left", validate="one_to_one",
    )
    result["cost"] = result["cost"].fillna(0)
    result["utm_campaign"] = result["utm_campaign"].fillna("none")
    return (
        result[["date_group", "visits", "registrations", "cost", "utm_campaign"]]
        .sort_values("date_group").reset_index(drop=True)
    )


ads_result = combine_advertising(conversion, ads)
ads_result.head(10)

,date_group,visits,registrations,cost,utm_campaign
0,2023-03-01,376,87,212.0,advanced_algorithms_series
1,2023-03-02,613,106,252.0,advanced_algorithms_series
2,2023-03-03,683,107,202.0,advanced_algorithms_series
3,2023-03-04,647,159,223.0,advanced_algorithms_series
4,2023-03-05,707,115,265.0,advanced_algorithms_series
5,2023-03-06,1291,230,108.0,advanced_algorithms_series
6,2023-03-07,1382,124,165.0,advanced_algorithms_series
7,2023-03-08,1382,151,155.0,advanced_algorithms_series
8,2023-03-09,1064,209,124.0,advanced_algorithms_series
9,2023-03-10,812,112,276.0,advanced_algorithms_series


## Проверки и сохранение

Проверяем, что объединение не размножило визиты и регистрации, а итоговые затраты
совпадают с расходами исходного CSV за даты метрик. Нулевые затраты существующей
кампании не означают отсутствие рекламы: её название сохраняется.

In [3]:
ads_in_period = ads["date_group"].isin(ads_result["date_group"])
assert ads_result["date_group"].is_monotonic_increasing
assert ads_result["date_group"].is_unique
assert ads_result["visits"].sum() == conversion["visits"].sum()
assert ads_result["registrations"].sum() == conversion["registrations"].sum()
assert abs(ads_result["cost"].sum() - ads.loc[ads_in_period, "cost"].sum()) < 1e-8

pd.Series({
    "Строк рекламы в CSV": len(ads),
    "Дней в результате": len(ads_result),
    "Дней без рекламы": int(ads_result["utm_campaign"].eq("none").sum()),
    "Затраты за даты метрик": ads_result["cost"].sum(),
    "Затраты вне дат метрик": ads.loc[~ads_in_period, "cost"].sum(),
}, name="Итого")

Строк рекламы в CSV         159.0
Дней в результате           184.0
Дней без рекламы             42.0
Затраты за даты метрик    27534.0
Затраты вне дат метрик     3445.0
Name: Итого, dtype: float64

In [4]:
ads_result.to_json("./ads.json", orient="columns", date_format="epoch", date_unit="ms")
print(f"ads.json сохранён: {len(ads_result)} строк, {len(ads_result.columns)} столбцов")

ads.json сохранён: 184 строк, 5 столбцов


/var/folders/m7/35gwjyss2wb45fxn4yvdxbm80000gn/T/ipykernel_15427/1506221049.py:1: Pandas4Warning: 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  ads_result.to_json("./ads.json", orient="columns", date_format="epoch", date_unit="ms")
